# F01 — ML Factor Model: Gu, Kelly & Xiu (2020)


## 0. Paper Information
| | |
|---|---|
| **Paper** | Empirical asset pricing via machine learning |
| **Authors** | Gu, S., Kelly, B., & Xiu, D. (2020) |
| **Venue** | Review of Financial Studies, 33(5), 2223-2273 |
| **Key Contribution** | ML significantly outperforms OLS in return prediction |
| **This notebook** | Cross-sectional return prediction with ENET / RF / GBM |


---
## 1. Research Design
Monthly cross-sectional prediction: $r_{i,t+1} = g(z_{i,t}) + \varepsilon_{i,t+1}$
where $z_{i,t}$ = 94 stock characteristics from the literature.
OOS R² (Campbell & Thompson 2008): benchmark is zero expected return.


---
## 3. Data


In [ ]:
from empirlab.finance import MLFactorModel
from empirlab.utils.metrics import ic, rmse
import numpy as np, pandas as pd, matplotlib.pyplot as plt

# Simulate CRSP-style characteristics (replace with real data)
rng = np.random.default_rng(42)
N, T, P = 500, 120, 20   # stocks, months, characteristics
chars   = rng.standard_normal((N * T, P))
returns = chars[:, 0] * 0.3 + rng.standard_normal(N * T) * 0.5
split   = int(N * T * 0.8)
X_tr, y_tr = chars[:split], returns[:split]
X_te, y_te = chars[split:], returns[split:]
print(f'Train: {X_tr.shape}, Test: {X_te.shape}')


---
## 4. Estimation


In [ ]:
results = {}
for method in ['enet', 'ridge', 'forest']:
    m = MLFactorModel(method=method).fit(X_tr, y_tr)
    r2 = m.r2_oos(X_te, y_te)
    ic_val = ic(m.predict(X_te), y_te)
    results[method] = {'OOS R2': r2, 'IC': ic_val}
    print(f'{method:8s} | OOS R²={r2:.4f} | IC={ic_val:.4f}')


---
## 5. Results & Robustness


In [ ]:
pd.DataFrame(results).T.round(4)


---
## 6. Visualization


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
methods = list(results)
r2s = [results[m]['OOS R2'] for m in methods]
ics  = [results[m]['IC']     for m in methods]
axes[0].bar(methods, r2s, color='steelblue')
axes[0].set_title('OOS R² by Method'); axes[0].set_ylabel('R²')
axes[1].bar(methods, ics,  color='coral')
axes[1].set_title('IC by Method'); axes[1].set_ylabel('Pearson IC')
plt.tight_layout(); plt.show()
